In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd
import numpy as np

import os
import pandas as pd
import numpy as np

#from keras.layers.wrappers import Bidirectional

In [3]:
data=pd.read_csv('/content/drive/MyDrive/HMPV Sentiment analysis/sentiment_output (1).csv')[['Cleaned_Translated_Comment','Sentiment_Label']]
data.dropna()
data

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/HMPV Sentiment analysis/sentiment_output (1).csv'

In [ ]:
'''test=pd.read_csv('/content/drive/MyDrive/Math Task/Math entity Relationship/test_data.csv')
test.dropna()
test'''

In [ ]:
character_set_not =  ['\n', '!', '"', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', ':', ';', '=', '?',
                      '[', '\\', ']', '{', '}', '\xa0', '¬', '´', '·', '।', '\u09e4', '৷', '\u200d', '–', '—', '‘', '’', '“', '”', '•']

In [ ]:
def tokenize(s):
    for i in character_set_not:
        s =  s.replace(i, '')
    s=s.lower()
    return  s

In [ ]:
'''train['Text']=train['Text'].map(tokenize)
train'''

In [ ]:
data['Cleaned_Translated_Comment']=data['Cleaned_Translated_Comment'].map(tokenize)
data

In [ ]:
labels = [0,1,2]
labels

In [ ]:
data.columns

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Load your dataset and split it into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(data['Cleaned_Translated_Comment'],data['Sentiment_Label'], test_size=0.2, random_state=42)

'''X_train= train['Text']
X_val= test['Text']
y_val= test['Relation']
y_train= train['Relation']'''

In [ ]:
X_val

In [ ]:
from transformers import XLNetTokenizerFast

# Load Tokenizer
tokenizer = XLNetTokenizerFast.from_pretrained("xlnet-base-cased")

# Define custom tokens to add
# new_tokens = ["$", "#"]

# Add the tokens to the tokenizer vocabulary (if you want to add custom tokens)
# num_added_toks = tokenizer.add_tokens(new_tokens)

# Tokenizing
train_encodings = tokenizer(list(X_train), truncation=True, padding=True)
test_encodings = tokenizer(list(X_val), truncation=True, padding=True)


In [ ]:
X_train

In [ ]:
X_train[860]

In [ ]:
y_train[860]

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(y_train)

y_train = label_encoder.transform(y_train)
y_test = label_encoder.transform(y_val)
y_train

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Assuming y_train and y_val are your original categorical labels
label_encoder = LabelEncoder()

# Fit the encoder on the training labels
label_encoder.fit(y_train)

# Show the mapping of original labels to encoded labels
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

# Print the label mappings
print("Label Encoder Mapping:")
for original_label, encoded_label in label_mapping.items():
    print(f"Original Label: {original_label} -> Encoded Label: {encoded_label}")


In [ ]:

num_labels = len(label_mapping)


In [ ]:
from transformers import XLNetTokenizerFast, TFXLNetForSequenceClassification
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
import numpy as np

# Load XLNet Tokenizer
'''tokenizer = XLNetTokenizerFast.from_pretrained("xlnet-base-cased")

# Tokenizing
train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=512, return_tensors="tf")
test_encodings = tokenizer(list(X_val), truncation=True, padding=True, max_length=512, return_tensors="tf")

# Label Encoding
label_encoder = LabelEncoder()
label_encoder.fit(y_train)

# Encode the labels
y_train = label_encoder.transform(y_train)
y_test = label_encoder.transform(y_val)

# Display label mapping
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label Encoder Mapping:")
for original_label, encoded_label in label_mapping.items():
    print(f"Original Label: {original_label} -> Encoded Label: {encoded_label}")'''

# Load XLNet Model for Sequence Classification
num_labels = len(label_mapping)
model = TFXLNetForSequenceClassification.from_pretrained("xlnet-base-cased", num_labels=num_labels)


# Optimizer, Loss, and Metrics
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-08, clipnorm=1.0)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metric = tf.keras.metrics.SparseCategoricalAccuracy("accuracy")

model.compile(optimizer=optimizer, loss=loss, metrics=[metric])
model.summary()

# Create TensorFlow Datasets
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encodings), np.array(y_train)))
test_dataset = tf.data.Dataset.from_tensor_slices((dict(test_encodings), np.array(y_test)))
bs=4

# Training the model
model.fit(
    train_dataset.batch(bs),
    epochs=1,  # Adjust epochs as needed
    validation_data=test_dataset.batch(bs)
)

# Predict on the test dataset
predictions = model.predict(test_dataset.batch(4))

# Get predicted class labels (from logits)
y_pred = tf.argmax(predictions.logits, axis=-1).numpy()

# Calculate Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')  # Weighted for multiclass
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# Print metrics
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)



In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encodings),np.array(y_train)))
test_dataset  = tf.data.Dataset.from_tensor_slices((dict(test_encodings),np.array(y_test)))

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
#es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=2)

model.fit(
    train_dataset.batch(8),
    epochs=1,
    batch_size=8,
    validation_data=test_dataset.batch(8),
    #callbacks=[es
)

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

# Assuming you have your test_dataset and true labels ready
test_dataset = tf.data.Dataset.from_tensor_slices((dict(test_encodings), np.array(y_val))).batch(12)

# Get the true labels and ensure they are numerical using label encoding
y_true = label_encoder.transform(y_val) # Use label_encoder to transform y_val into numerical labels

# Make predictions on the test dataset
y_pred = model.predict(test_dataset).logits
y_pred = np.argmax(y_pred, axis=1)

# Calculate the confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:\n", conf_matrix)


In [ ]:
classes=['Negative','Neutral','Positive']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# ... (your existing code to calculate the confusion matrix) ...

# Create a heatmap of the confusion matrix
plt.figure(figsize=(10, 7))  # Adjust figure size as needed
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="crest",
            xticklabels=classes,  # Use original labels for x-axis
            yticklabels=classes,linewidths=1)  # Use original labels for y-axis
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix Heatmap")
plt.savefig('/content/drive/MyDrive/HMPV Sentiment analysis/Figures/xlnetcm.pdf')
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ... (your existing code to calculate y_true and y_pred) ...

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred)

# Calculate precision, recall, and F1-score (macro-averaged)
precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

# Print the results
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (macro): {precision:.4f}")
print(f"Recall (macro): {recall:.4f}")
print(f"F1-score (macro): {f1:.4f}")

In [ ]:
id2labels = model.config.id2label
model.config.id2label = {id : label_encoder.inverse_transform([id])[0]  for id, label in id2labels.items()}

label2ids = model.config.label2id
model.config.label2id = {label_encoder.inverse_transform([id])[0] : id   for id, label in id2labels.items()}

In [ ]:
!pip install shap

In [ ]:
import transformers
import shap

In [ ]:
pred = transformers.pipeline("text-classification", model=model, tokenizer=tokenizer, device=0, return_all_scores=True)
explainer = shap.Explainer(pred)

In [ ]:
ls=[
    "My parents are non believers when it comes with viruses so i dont know how i think of this",#Neu
    "Depending on your religion, and maybe even your intellect, metaphysically speaking, this might be one of those new viral pandemics being spread by Uranus, or by Jupiter or perhaps even Mars, or, god help us, Saturn ~ The god's must be going mad again...  (Nevermind, Trump will save America, and then America will save the rest of us)...", #Pos
    'This is not real news. The Hmpv virus is the common flu.  No proof of anything and no confirmation on the ground and they still report on it !!  This is just bad journalism, no one serious is reporting on this anywhere around the world â€¦.  This news is pure FEAR MONGERING' #Neg
]
ls=[tokenize(s) for s in ls]

In [ ]:
shap_values = explainer(ls)

In [ ]:
classes=['Negative','Neutral','Positive']

In [ ]:
shap_values.output_names=classes
shap_values.output_names

In [ ]:
shap.plots.text(shap_values, display=True)

In [ ]:
import matplotlib.pyplot as plt
from IPython.core.display import HTML
HTML(shap.plots.text(shap_values, display=True))
with open('/content/drive/MyDrive/HMPV Sentiment analysis/Figures/shapfinal.html', 'w') as file:
     file.write(shap.plots.text(shap_values, display=False))

In [ ]:
import matplotlib.pyplot as plt
from IPython.core.display import HTML
HTML(shap.plots.text(shap_values[:,:,classes[1]], display=True))
with open('/content/drive/MyDrive/HMPV Sentiment analysis/Figures/shap classwiseNeutral.html', 'w') as file:
     file.write(shap.plots.text(shap_values[:,:,classes[1]], display=False))

In [ ]:
shap.initjs
plt.figure(figsize=(9,13))
shap.plots.bar(shap_values[:,:,classes[0]].mean(0),max_display=20)
plt.savefig('/content/drive/MyDrive/HMPV Sentiment analysis/Figures/shapvaluesbar.pdf')
plt.close()

In [ ]:
shap.plots.bar(shap_values[:,:,classes[1]].mean(0),max_display=20)

In [ ]:
shap.plots.bar(shap_values[:,:,classes[2]].mean(0),max_display=20)

In [ ]:
# we can sort the bar chart in decending order
shap.plots.bar(shap_values[:, :, classes[0]].mean(0), order=shap.Explanation.argsort,max_display=30)

In [ ]:
shap.plots.bar(shap_values[:, :, classes[3]].mean(0), order=shap.Explanation.argsort.flip,max_display=20)

In [ ]:
shap.waterfall_plot(shap_values[0][:, classes[1]], max_display=15)

In [ ]:
logit_explainer = shap.Explainer(shap.models.TransformersPipeline(pred, rescale_to_logits=True))

logit_shap_values = logit_explainer(test["Text"][:3])
shap.plots.text(logit_shap_values)

In [ ]:
plt.rcParams["font.family"] = "Arial Unicode MS"  # Use a font that supports Bangla characters

# Plot the bar chart with Bangla feature names
shap.plots.bar(shap_values[:, :, '2'].mean(0))



# Display plot


In [ ]:
shap_values.feature_names